In [4]:
# ============================================================
# HOMEWORK 2 - PROBLEM 1
# Linear Regression using Gradient Descent from Scratch
# ============================================================

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from sklearn.model_selection import train_test_split


# ============================================================
# MOUNT GOOGLE DRIVE AND LOAD DATA
# ============================================================

drive.mount('/content/drive')

file_path = '/content/drive/MyDrive/Housing.csv'

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())


# ============================================================
# CONVERT YES/NO VARIABLES TO 1/0
# Needed for Problem 1b
# ============================================================

yes_no_columns = [
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'prefarea'
]

for column in yes_no_columns:
    if column in df.columns:
        df[column] = df[column].map({
            'yes': 1,
            'no': 0
        })


# ============================================================
# LOSS FUNCTION
# ============================================================

def compute_loss(X, y, theta):

    m = len(y)

    predictions = X @ theta

    errors = predictions - y

    loss = (1 / (2 * m)) * np.sum(errors ** 2)

    return loss


# ============================================================
# GRADIENT DESCENT FROM SCRATCH
# ============================================================

def gradient_descent(
    X_train,
    y_train,
    X_val,
    y_val,
    learning_rate,
    iterations
):

    # Number of training samples
    m = len(y_train)

    # Add column of ones for theta_0
    X_train_bias = np.c_[
        np.ones(m),
        X_train
    ]

    X_val_bias = np.c_[
        np.ones(len(y_val)),
        X_val
    ]

    # Initialize theta values to zero
    theta = np.zeros(
        X_train_bias.shape[1]
    )

    train_losses = []
    val_losses = []

    for i in range(iterations):

        # Prediction
        predictions = X_train_bias @ theta

        # Error
        errors = predictions - y_train

        # Gradient
        gradient = (
            1 / m
        ) * (
            X_train_bias.T @ errors
        )

        # Gradient descent update
        theta = (
            theta
            -
            learning_rate * gradient
        )

        # Compute training loss
        train_loss = compute_loss(
            X_train_bias,
            y_train,
            theta
        )

        # Compute validation loss
        val_loss = compute_loss(
            X_val_bias,
            y_val,
            theta
        )

        train_losses.append(
            train_loss
        )

        val_losses.append(
            val_loss
        )

        # Stop if divergence occurs
        if (
            not np.isfinite(train_loss)
            or
            not np.isfinite(val_loss)
        ):
            break

    return (
        theta,
        train_losses,
        val_losses
    )


# ============================================================
# FUNCTION FOR RUNNING ONE PROBLEM
# ============================================================

def run_experiment(
    X,
    y,
    feature_names,
    problem_name,
    learning_rates,
    iterations
):

    # ========================================================
    # 80/20 TRAIN-VALIDATION SPLIT
    # ========================================================

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42
    )

    # Convert to float arrays
    X_train = np.array(
        X_train,
        dtype=float
    )

    X_val = np.array(
        X_val,
        dtype=float
    )

    y_train = np.array(
        y_train,
        dtype=float
    )

    y_val = np.array(
        y_val,
        dtype=float
    )

    print("\n")
    print("=" * 70)
    print(problem_name)
    print("=" * 70)

    print(
        "Training samples:",
        len(X_train)
    )

    print(
        "Validation samples:",
        len(X_val)
    )

    print(
        "Iterations:",
        iterations
    )

    print(
        "Learning rates tested:",
        learning_rates
    )

    # Store all results
    results = {}

    # ========================================================
    # TEST DIFFERENT LEARNING RATES
    # ========================================================

    for alpha in learning_rates:

        theta, train_losses, val_losses = gradient_descent(
            X_train,
            y_train,
            X_val,
            y_val,
            learning_rate=alpha,
            iterations=iterations
        )

        final_train_loss = train_losses[-1]
        final_val_loss = val_losses[-1]

        results[alpha] = {
            'theta': theta,
            'train_losses': train_losses,
            'val_losses': val_losses,
            'final_train_loss': final_train_loss,
            'final_val_loss': final_val_loss
        }

        print("\nLearning Rate =", alpha)

        print(
            "Final Training Loss =",
            final_train_loss
        )

        print(
            "Final Validation Loss =",
            final_val_loss
        )


    # ========================================================
    # SELECT BEST LEARNING RATE
    # Based on lowest final validation loss
    # ========================================================

    valid_results = {
        alpha: result
        for alpha, result in results.items()
        if np.isfinite(
            result['final_val_loss']
        )
    }

    if len(valid_results) == 0:

        print(
            "\nAll tested learning rates diverged."
        )

        return None

    best_alpha = min(
        valid_results,
        key=lambda alpha:
        valid_results[alpha][
            'final_val_loss'
        ]
    )

    best_result = valid_results[
        best_alpha
    ]

    best_theta = best_result[
        'theta'
    ]

    best_train_losses = best_result[
        'train_losses'
    ]

    best_val_losses = best_result[
        'val_losses'
    ]

    # ========================================================
    # PRINT BEST RESULT
    # ========================================================

    print("\n")
    print("-" * 70)

    print(
        "BEST RESULT FOR",
        problem_name
    )

    print("-" * 70)

    print(
        "Best Learning Rate:",
        best_alpha
    )

    print(
        "Final Training Loss:",
        best_result[
            'final_train_loss'
        ]
    )

    print(
        "Final Validation Loss:",
        best_result[
            'final_val_loss'
        ]
    )


    # ========================================================
    # PRINT PARAMETERS
    # ========================================================

    print("\nBest Parameters:")

    print(
        "theta_0 (intercept) =",
        best_theta[0]
    )

    for i, feature in enumerate(
        feature_names
    ):

        print(
            f"theta_{i+1} ({feature}) =",
            best_theta[i+1]
        )


    # ========================================================
    # PLOT TRAINING AND VALIDATION LOSSES
    # ========================================================

    plt.figure(
        figsize=(9, 6)
    )

    plt.plot(
        range(
            1,
            len(best_train_losses) + 1
        ),
        best_train_losses,
        label='Training Loss'
    )

    plt.plot(
        range(
            1,
            len(best_val_losses) + 1
        ),
        best_val_losses,
        label='Validation Loss'
    )

    plt.xlabel(
        'Iteration'
    )

    plt.ylabel(
        'Loss'
    )

    plt.title(
        f'{problem_name}: Training and Validation Loss\n'
        f'Learning Rate = {best_alpha}'
    )

    plt.legend()

    plt.grid()

    plt.show()


    # Return best values for comparison
    return {
        'best_alpha': best_alpha,
        'theta': best_theta,
        'train_loss':
            best_result[
                'final_train_loss'
            ],
        'val_loss':
            best_result[
                'final_val_loss'
            ]
    }


# ============================================================
# LEARNING RATES
# Explore values between 0.01 and 0.1
# ============================================================

learning_rates = [
    0.01,
    0.025,
    0.05,
    0.075,
    0.1
]

# Chosen number of iterations
iterations = 1000


# ============================================================
# PROBLEM 1a
# ============================================================

features_1a = [
    'area',
    'bedrooms',
    'bathrooms',
    'stories',
    'parking'
]

X_1a = df[
    features_1a
].values

y = df[
    'price'
].values

result_1a = run_experiment(
    X=X_1a,
    y=y,
    feature_names=features_1a,
    problem_name='Problem 1a',
    learning_rates=learning_rates,
    iterations=iterations
)


# ============================================================
# PROBLEM 1b
# ============================================================

features_1b = [
    'area',
    'bedrooms',
    'bathrooms',
    'stories',
    'mainroad',
    'guestroom',
    'basement',
    'hotwaterheating',
    'airconditioning',
    'parking',
    'prefarea'
]

X_1b = df[
    features_1b
].values

result_1b = run_experiment(
    X=X_1b,
    y=y,
    feature_names=features_1b,
    problem_name='Problem 1b',
    learning_rates=learning_rates,
    iterations=iterations
)


# ============================================================
# COMPARE PROBLEM 1a AND PROBLEM 1b
# ============================================================

if (
    result_1a is not None
    and
    result_1b is not None
):

    print("\n")
    print("=" * 70)
    print("COMPARISON: PROBLEM 1a VS PROBLEM 1b")
    print("=" * 70)

    comparison = pd.DataFrame({

        'Problem': [
            '1a',
            '1b'
        ],

        'Number of Features': [
            len(features_1a),
            len(features_1b)
        ],

        'Best Learning Rate': [
            result_1a[
                'best_alpha'
            ],
            result_1b[
                'best_alpha'
            ]
        ],

        'Final Training Loss': [
            result_1a[
                'train_loss'
            ],
            result_1b[
                'train_loss'
            ]
        ],

        'Final Validation Loss': [
            result_1a[
                'val_loss'
            ],
            result_1b[
                'val_loss'
            ]
        ]

    })

    display(comparison)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset shape: (545, 13)

Columns:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']

First five rows:


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished




Problem 1a
Training samples: 436
Validation samples: 109
Iterations: 1000
Learning rates tested: [0.01, 0.025, 0.05, 0.075, 0.1]

Learning Rate = 0.01
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate = 0.025
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate = 0.05
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate = 0.075
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate = 0.1
Final Training Loss = inf
Final Validation Loss = inf

All tested learning rates diverged.


Problem 1b
Training samples: 436
Validation samples: 109
Iterations: 1000
Learning rates tested: [0.01, 0.025, 0.05, 0.075, 0.1]

Learning Rate = 0.01
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate = 0.025
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate = 0.05
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate = 0.075
Final Training Loss = inf
Final Validation Loss = inf

Learning Rate

/tmp/ipykernel_1500/2145163293.py:71: RuntimeWarning: overflow encountered in square
  loss = (1 / (2 * m)) * np.sum(errors ** 2)
/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
/tmp/ipykernel_1500/2145163293.py:71: RuntimeWarning: overflow encountered in square
  loss = (1 / (2 * m)) * np.sum(errors ** 2)
/usr/local/lib/python3.13/dist-packages/numpy/_core/fromnumeric.py:86: RuntimeWarning: overflow encountered in reduce
  return ufunc.reduce(obj, axis, dtype, out, **passkwargs)
